# Pasado `gpu-hybrid`: v3 (pivot-centriran z) i kompajlirani `planar_fit` na CUDA-i

Pre pokretanja: **Runtime > Change runtime type > GPU (A100)**, zatim **Runtime > Disconnect and delete runtime**, pa **Runtime > Run all**.

Kod na commitu `@@COMMIT@@`: ispravka centriranja z (`500ae16`), v3 poređenja na CPU-u (`results/lstsq_comparison_2026-10-27_v3_zpivot/`) i opt-in kompajlirani put (`PASADO_COMPILED_FIT`, `planar_fit.compiled_fit_planes`, sa fallback-om na eager).

Redom (jedno za drugim na istom GPU-u):
1. **Hladan i topao start kompilacije**, pre svega ostalog: dva odvojena procesa sa istim, na početku praznim Inductor kešom na disku; za `compile` i `reduce-overhead`.
2. **R4 za v3**: `--level timing --device cuda` (kod A2 se promenio, stari timing ne važi).
3. **Korektnost na CUDA-i**: `reproduce_target_centring.py`, `test_planar_fit.py` (eager i `--compiled`), `test_batched_splits.py` (gelsd, `--closed-form`, `--compiled`).
4. **Aliasing kod `reduce-overhead`**: da li sledeći poziv prepiše prethodni izlaz, i da li batched prolaz bez kopije daje iste granice.
5. **End-to-end**: sekvencijalni G, batched G, A2 eager, A2 `compile`, A2 `reduce-overhead`; 4 mreže, `eps_idx` 0 i 8, `n_splits` 1, 8, 32, 64, 3 slike; uz vreme prvog prolaza.
6. **Break-even**: koliko prolaza treba da se kompilacija (hladna i topla) isplati, po konfiguraciji.

Očekivano trajanje: oko 45 do 75 minuta (procena). Kad završi, pošalji `v3_compiled_cuda.zip`.

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"
GPU = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", GPU,
      "| VRAM GiB:", round(VRAM_GIB, 1))
REQUIRE_A100 = True
assert "A100" in GPU or not REQUIRE_A100, f"Merenje je dogovoreno na A100, a ovo je {GPU}."

In [ ]:
import os
import shutil

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
COMMIT = "832323a"

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--branch", "gpu-hybrid",
                "--single-branch", URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head.startswith(COMMIT), f"HEAD je {head}, a ocekivan je {COMMIT}"
print("HEAD:", head)
os.chdir(REPO)

OUT = "/content/v3_compiled_cuda"
os.makedirs(OUT, exist_ok=True)

In [ ]:
%pip install -q scikit-learn mpmath

In [ ]:
def run(cmd, log, env=None):
    """A shell command with its output on screen and in a log file; returns the exit code."""
    script = f'''
set -o pipefail
PYTHONUNBUFFERED=1 {cmd} 2>&1 | tee {log}
'''
    return subprocess.run(["bash", "-lc", script], cwd=REPO, env={**os.environ, **(env or {})}).returncode

## 1. Hladan i topao start kompilacije

Svaki režim dobija svoj, na početku prazan keš (`TORCHINDUCTOR_CACHE_DIR`). Prvi proces je hladan; drugi proces, sa istim kešom, je topao (Inductor/FX keš na disku). Meri se vreme prvog poziva za `n = 100`, zatim `1024` i `65536` (posle drugog oblika `n` postaje dinamičan, pa treći oblik ne bi trebalo da kompajlira).

In [ ]:
cold_warm_script = f'''
import json, sys, time, torch
sys.path.insert(0, "{REPO}/forward_mode_tensorized_src"); sys.path.insert(0, "{REPO}/experiments")
t_start = time.perf_counter()
import planar_fit, compare_lstsq_solvers as h
mode = None if sys.argv[1] == "default" else sys.argv[1]
fn = planar_fit.compiled_fit_planes(mode)
out = dict(mode=sys.argv[1], run=sys.argv[2], import_s=time.perf_counter() - t_start)
with torch.no_grad():
    for n in (100, 1024, 65536):
        b = h.synthetic_bounds("normal", n, 3, "float64")
        g, z = h.grid_and_targets(*(t.to("cuda") for t in b))
        torch.cuda.synchronize(); t0 = time.perf_counter()
        fn(g, z)
        torch.cuda.synchronize()
        out[f"first_call_s_n{{n}}"] = time.perf_counter() - t0
out["fallback"] = planar_fit.compile_fallback
out["graphs"] = torch._dynamo.utils.counters["stats"].get("unique_graphs", 0)
print("RESULT " + json.dumps(out))
'''
with open("/content/cold_warm.py", "w") as f:
    f.write(cold_warm_script)

import json
start_rows = []
for mode in ("default", "reduce-overhead"):
    cache = f"/content/inductor_cache_{mode}"
    shutil.rmtree(cache, ignore_errors=True)
    for run_name in ("cold", "warm"):
        p = subprocess.run(["python", "/content/cold_warm.py", mode, run_name], cwd=REPO, text=True,
                           capture_output=True, env={**os.environ, "TORCHINDUCTOR_CACHE_DIR": cache})
        line = [l for l in p.stdout.splitlines() if l.startswith("RESULT ")]
        assert p.returncode == 0 and line, p.stdout[-2000:] + p.stderr[-2000:]
        row = json.loads(line[0][7:])
        start_rows.append(row)
        print(row, flush=True)

import pandas as pd
from IPython.display import display
start = pd.DataFrame(start_rows)
start.to_csv(f"{OUT}/cold_warm_start.csv", index=False)
display(start)

## 2. R4 za v3: vreme solvera na A100

Isti nivo `timing` pravila kao 27.10, sada sa kodom posle ispravke. `timing_cuda.csv` iz ovog foldera ide u v3 folder za odluku po pravilu.

In [ ]:
rc = run(f"python experiments/compare_lstsq_solvers.py --level timing --device cuda --out {OUT}",
         f"{OUT}/timing_level.log")
assert rc == 0, "timing nije prošao; pošalji timing_level.log"
t = pd.read_csv(f"{OUT}/timing_cuda.csv")
display(t[t.dtype == "float64"].pivot(index="candidate", columns="n", values="median_ms").round(3))

## 3. Korektnost na CUDA-i

In [ ]:
checks = [
    ("python experiments/reproduce_target_centring.py --device cuda", "reproduce_target_centring_cuda.log"),
    ("python experiments/test_planar_fit.py --device cuda", "test_planar_fit_cuda.log"),
    ("python experiments/test_planar_fit.py --device cuda --compiled", "test_planar_fit_compiled_cuda.log"),
    ("python experiments/test_batched_splits.py --device cuda", "test_batched_splits_gelsd_cuda.log"),
    ("python experiments/test_batched_splits.py --device cuda --closed-form", "test_batched_splits_closed_form_cuda.log"),
    ("python experiments/test_batched_splits.py --device cuda --compiled", "test_batched_splits_compiled_cuda.log"),
]
results = {}
for cmd, log in checks:
    results[log] = run(cmd, f"{OUT}/{log}")
print(results)
assert all(rc == 0 for rc in results.values()), "neki test nije prošao; pošalji logove iz OUT"

## 4. Aliasing kod `reduce-overhead`

CUDA graph zadržava izlazne bafere. Prvo: da li drugi poziv istog oblika prepiše izlaz prvog poziva koji je sačuvan bez kopije. Zatim: batched prolaz sa `reduce-overhead` bez kopije izlaza mora da da iste granice kao eager zatvorena forma, jer `precise_sigmoid` koristi koeficijente pre sledećeg poziva.

In [ ]:
import math
import statistics
import sys
import time

for p in (f"{REPO}/forward_mode_tensorized_src", f"{REPO}/Section_5_4", f"{REPO}/experiments"):
    if p not in sys.path:
        sys.path.insert(0, p)

import batched_splits as bs
import planar_fit
import precise_transformer as pt
from compare_lstsq_solvers import (EPS_SCHEDULE, grid_and_targets, set_production_flags,
                                   swapped_solver, synthetic_bounds)
from measure_split_sweep import build_forward_precise, eval_splits_once, load_network_and_images

set_production_flags()
torch.set_default_dtype(torch.float64)
device = torch.device("cuda")
alias = {}
with torch.no_grad():
    ro = planar_fit.compiled_fit_planes("reduce-overhead")
    g1, z1 = grid_and_targets(*(t.to(device) for t in synthetic_bounds("normal", 1024, 21, "float64")))
    g2, z2 = grid_and_targets(*(t.to(device) for t in synthetic_bounds("normal", 1024, 22, "float64")))
    for _ in range(3):  # warm up the CUDA graph
        ro(g1, z1)
    out1 = ro(g1, z1)
    kept = out1.clone()
    out2 = ro(g2, z2)
    alias["first_output_overwritten"] = not torch.equal(out1, kept)
    alias["first_output_now_equals_second"] = bool(torch.equal(out1, out2))
    net, images = load_network_and_images("big", device, torch.float64, 1)
    layers = bs.extract_affine_layers(net, "big")
    img = images[0][1]
    eager = bs.lipschitz_bound(layers, img, EPS_SCHEDULE[0], 8, seed=7, chunk=8)  # gelsd
    bs.USE_CLOSED_FORM_FIT = True
    ref = bs.lipschitz_bound(layers, img, EPS_SCHEDULE[0], 8, seed=7, chunk=8)
    with swapped_solver(lambda g, z: ro(g, z)):  # no copy of the outputs
        bs.USE_CLOSED_FORM_FIT = False
        got = bs.lipschitz_bound(layers, img, EPS_SCHEDULE[0], 8, seed=7, chunk=8)
alias["bound_gelsd"], alias["bound_eager_closed_form"], alias["bound_reduce_overhead_no_copy"] = eager, ref, got
alias["rel_diff_no_copy_vs_eager"] = abs(got - ref) / abs(ref)
print(alias)
with open(f"{OUT}/aliasing.json", "w") as f:
    json.dump(alias, f, indent=2)
assert alias["rel_diff_no_copy_vs_eager"] <= 1e-12, "batched prolaz bez kopije ne daje istu granicu"

## 5. End-to-end

Sve u istom runu, seed 7. Prvi prolaz za svaku konfiguraciju je warm-up (uključuje kompilaciju za nove oblike u ovom procesu; keš je u ovom trenutku već topao od sekcija 1 i 3) i beleži se posebno. Pozivi solvera se broje.

In [ ]:
SPLITS = [1, 8, 32, 64]
EPS_INDICES = [0, 8]
NUM_IMAGES = 3
REPEATS = 3
BIG_CHUNK = min(64, 2 ** int(math.log2(max(1, 0.6 * VRAM_GIB / 0.5))))
calls = {"eager": 0, "swapped": 0}
fit_planes_eager = planar_fit.fit_planes


def counted_eager(grid, z, rcond=None):
    calls["eager"] += 1
    return fit_planes_eager(grid, z, rcond)


planar_fit.fit_planes = counted_eager
# compiled from the unpatched function, so the call counter stays outside the graph
compiled_default = torch.compile(fit_planes_eager, fullgraph=True)
compiled_ro = torch.compile(fit_planes_eager, fullgraph=True, mode="reduce-overhead")


def swapped(fn):
    def run_fn(grid, z):
        calls["swapped"] += 1
        return fn(grid, z)
    return run_fn


MODES = ["sequential_G", "batched_G", "batched_A2_eager", "batched_A2_compile", "batched_A2_reduce_overhead"]
rows = []
with torch.no_grad():
    for network in ["3layer", "4layer", "5layer", "big"]:
        net, images = load_network_and_images(network, device, torch.float64, NUM_IMAGES)
        forward_fn = build_forward_precise(pt, net, network)
        layers = bs.extract_affine_layers(net, network)
        width = 1024 if network == "big" else 100
        chunk = BIG_CHUNK if network == "big" else None
        for image_id, img in images:
            for eps_idx in EPS_INDICES:
                eps = EPS_SCHEDULE[eps_idx]
                for n_splits in SPLITS:
                    row = dict(network=network, image_id=image_id, eps_idx=eps_idx, n_splits=n_splits,
                               boxes_per_call=n_splits * width)
                    for mode in MODES:
                        bs.USE_CLOSED_FORM_FIT = mode == "batched_A2_eager"
                        solver = {"batched_A2_compile": swapped(compiled_default),
                                  "batched_A2_reduce_overhead": swapped(compiled_ro)}.get(mode)
                        if mode == "sequential_G":
                            call = lambda: eval_splits_once(forward_fn, img, eps, n_splits, 7, device, torch.float64)
                        else:
                            call = lambda: bs.lipschitz_bound(layers, img, eps, n_splits, seed=7, chunk=chunk)
                        calls["eager"] = calls["swapped"] = 0
                        ctx = swapped_solver(solver) if solver else None
                        if ctx:
                            ctx.__enter__()
                        try:
                            torch.cuda.synchronize(); t0 = time.perf_counter()
                            call()
                            torch.cuda.synchronize()
                            row[f"first_ms_{mode}"] = (time.perf_counter() - t0) * 1000.0
                            times = []
                            for _ in range(REPEATS):
                                torch.cuda.synchronize(); t0 = time.perf_counter()
                                bound = call()
                                torch.cuda.synchronize()
                                times.append((time.perf_counter() - t0) * 1000.0)
                        finally:
                            if ctx:
                                ctx.__exit__(None, None, None)
                            bs.USE_CLOSED_FORM_FIT = False
                        assert (calls["eager"] > 0) == (mode == "batched_A2_eager"), (mode, calls)
                        assert (calls["swapped"] > 0) == (solver is not None), (mode, calls)
                        row[f"ms_{mode}"] = statistics.median(times)
                        row[f"bound_{mode}"] = bound
                    rows.append(row)
                print(f"{network} image {image_id} eps_idx {eps_idx} done", flush=True)
planar_fit.fit_planes = fit_planes_eager

e2e = pd.DataFrame(rows)
for mode in MODES[1:]:
    e2e[f"rel_diff_{mode}"] = (e2e[f"bound_{mode}"] - e2e.bound_sequential_G).abs() / e2e.bound_sequential_G.abs()
worst = e2e[[f"rel_diff_{m}" for m in MODES[1:]]].max()
print(worst.to_string())
e2e.to_csv(f"{OUT}/e2e.csv", index=False)
assert (worst <= 1e-9).all(), "granica se ne slaže sa sekvencijalnom; pošalji e2e.csv"
summary = e2e.groupby(["network", "eps_idx", "n_splits", "boxes_per_call"], as_index=False).agg(
    **{f"ms_{m}": (f"ms_{m}", "mean") for m in MODES})
for m in MODES[2:]:
    summary[f"{m}_vs_batched_G"] = summary.ms_batched_G / summary[f"ms_{m}"]
    summary[f"{m}_vs_sequential"] = summary.ms_sequential_G / summary[f"ms_{m}"]
summary.to_csv(f"{OUT}/e2e_summary.csv", index=False)
pd.set_option("display.width", 250)
display(summary.round(2))

## 6. Break-even

Prolazi potrebni da se kompilacija isplati u odnosu na batched G: `compile_s / (t_batched_G - t_mode)` po konfiguraciji, za hladan i topao start iz sekcije 1 (zbir prvih poziva za `n = 100` i `1024`, gde se kompajlira).

In [ ]:
be = []
for mode, col in (("default", "batched_A2_compile"), ("reduce-overhead", "batched_A2_reduce_overhead")):
    for run_name in ("cold", "warm"):
        r = start[(start["mode"] == mode) & (start["run"] == run_name)].iloc[0]
        compile_s = r["first_call_s_n100"] + r["first_call_s_n1024"]
        for _, s in summary[summary.eps_idx == 0].iterrows():
            save_ms = s.ms_batched_G - s[f"ms_{col}"]
            be.append(dict(mode=mode, start=run_name, compile_s=compile_s, network=s.network,
                           n_splits=s.n_splits, saves_ms_per_pass=save_ms,
                           passes_to_break_even=(compile_s * 1000.0 / save_ms) if save_ms > 0 else float("inf")))
be = pd.DataFrame(be)
be.to_csv(f"{OUT}/break_even.csv", index=False)
display(be.pivot_table(index=["mode", "start", "network"], columns="n_splits",
                       values="passes_to_break_even").round(0))

## 7. Preuzimanje rezultata (`.zip`)

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

with open(f"{OUT}/colab_environment.json", "w") as f:
    json.dump(dict(gpu=GPU, vram_gib=VRAM_GIB, torch=torch.__version__, cuda=torch.version.cuda,
                   commit=head, big_chunk=BIG_CHUNK, repeats=REPEATS), f, indent=2)
archive = "/content/v3_compiled_cuda.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(OUT)):
        zf.write(os.path.join(OUT, name), os.path.join("v3_compiled_cuda", name))
print(archive, round(os.path.getsize(archive) / 2**20, 2), "MiB")
files.download(archive)